# Ariel End-to-End Training Pipeline

Notebook này chạy pipeline hiện tại từ raw Kaggle data đến `submission.csv`:

1. Build train/test features từ parquet.
2. Join `train.csv` target.
3. Chọn model config.
4. Validate hoặc cross-validate.
5. Refit model trên full train.
6. Predict test và ghi submission.

Khuyến nghị: chạy smoke test với `RUN_MODE = "smoke"` trước. Khi shape/parquet ổn, đổi sang `RUN_MODE = "full"`.

## Environment Setup

Use the project `.venv` kernel, not the base Anaconda kernel. This avoids the old `numexpr`/`bottleneck` binary mismatch with NumPy 2.x.

From the repository root, run once in PowerShell:

```powershell
python -m venv .venv
.venv\Scripts\python.exe -m pip install --upgrade pip
.venv\Scripts\python.exe -m pip install -r requirements.txt
.venv\Scripts\python.exe -m pip install ipykernel
.venv\Scripts\python.exe -m ipykernel install --user --name ariel-ml --display-name "Ariel ML (.venv)"
```

Then select kernel `Ariel ML (.venv)` in Jupyter before running this notebook.


In [ ]:
import sys
from pathlib import Path

print(sys.executable)
if ".venv" not in sys.executable:
    print("WARNING: this notebook is not running from the project .venv kernel.")


## Configuration Guide

Các model tabular dùng feature matrix `X` và target spectrum `Y`:

| Model name | Ý nghĩa | Dependency | Khi dùng |
|---|---|---|---|
| `bayesian_ridge` | Model chính trong plan: Bayesian Ridge + PCA + calibrated sigma | built-in | baseline mạnh, ổn định |
| `ridge` | Ridge + PCA | built-in | sanity baseline nhanh |
| `kernel_ridge` | Kernel Ridge + PCA | built-in | nonlinear nhẹ, dataset nhỏ |
| `extra_trees` | ExtraTrees + PCA | built-in | tree baseline, robust outlier |
| `boosting` | sklearn GradientBoosting + PCA | built-in | fallback cho boosting tabular |
| `lightgbm` | LightGBM + PCA | optional | tabular mạnh, cần `requirements-optional.txt` |
| `xgboost` | XGBoost + PCA | optional | tabular mạnh, cần `requirements-optional.txt` |
| `br_boosting_residual` | Bayesian Ridge + sklearn boosting residual correction | built-in | thử hybrid không cần optional deps |
| `br_lgbm_residual` | Bayesian Ridge + LightGBM residual correction | optional | hybrid theo plan |

Các cấu hình chính:

- `N_COMPONENTS`: số PCA components của target. Plan gợi ý `20, 30, 40`. Smoke test nên dùng nhỏ hơn nếu sample ít.
- `TIME_BINS`: số bin thời gian sau preprocess. Dùng `20` cho smoke, `64` cho debug, `128` cho baseline thực tế; chỉ tăng lên `300` hoặc `500` khi đã chấp nhận thời gian/RAM.
- `RUN_SEARCH`: bật search model/PCA bằng grouped CV.
- `RUN_CV`: chạy CV riêng cho model đã chọn.
- `REFIT_FULL`: sau validation, train lại trên toàn bộ train rows để tạo submission model.
- `LIMIT`: số planet đọc vào. `5` cho smoke, `200` cho baseline nhanh, `None` cho full.

Deep learning baselines nằm trong `ariel_ml.deep_models`, nhưng notebook này tập trung vào tabular physics features vì đó là final recommendation trong plan. DL nên chạy riêng sau khi raw light-curve tensors đã được cache ổn định.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import joblib
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from ariel_ml.config import DatasetConfig, FeatureConfig, ModelConfig, PreprocessConfig
from ariel_ml.dataset_builder import ArielDatasetBuilder, align_features_and_targets
from ariel_ml.io import ArielDataRepository
from ariel_ml.pipeline import ArielPreprocessFeaturePipeline
from ariel_ml.submission import infer_submission_schema, predict_submission, save_submission
from ariel_ml.training import cross_validate_model, hyperparameter_search, refit_full_model, train_model

DATA_ROOT = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "model"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT, OUTPUT_DIR

## 0. Select Run Preset

Chọn một preset:

- `smoke`: đọc vài planet, dùng `ridge`, nhanh để kiểm tra IO/shape.
- `baseline`: chạy model chính `bayesian_ridge` với `N_COMPONENTS=30`.
- `search`: chạy grouped CV trên nhiều model và PCA components.
- `residual`: chạy hybrid `br_boosting_residual` hoặc `br_lgbm_residual`.
- `full`: giống baseline nhưng đọc toàn bộ data.

Nếu dùng `lightgbm`, `xgboost`, hoặc `br_lgbm_residual`, cài trước: `python -m pip install -r requirements-optional.txt`.

In [ ]:
RUN_MODE = "smoke"  # smoke | baseline | search | residual | full

PRESETS = {
    "smoke": {
        "limit": 5,
        "time_bins": 20,
        "model_name": "ridge",
        "n_components": 3,
        "run_search": False,
        "run_cv": False,
        "refit_full": False,
    },
    "baseline": {
        "limit": 200,
        "time_bins": 128,
        "model_name": "bayesian_ridge",
        "n_components": 30,
        "run_search": False,
        "run_cv": True,
        "refit_full": True,
    },
    "search": {
        "limit": 200,
        "time_bins": 128,
        "model_name": "bayesian_ridge",
        "n_components": 30,
        "run_search": True,
        "run_cv": False,
        "refit_full": True,
    },
    "residual": {
        "limit": 200,
        "time_bins": 128,
        "model_name": "br_boosting_residual",
        "n_components": 30,
        "run_search": False,
        "run_cv": True,
        "refit_full": True,
    },
    "full": {
        "limit": None,
        "time_bins": 128,
        "model_name": "bayesian_ridge",
        "n_components": 30,
        "run_search": False,
        "run_cv": False,
        "refit_full": True,
    },
}

cfg = PRESETS[RUN_MODE].copy()
RANDOM_STATE = 42
CV_SPLITS = 5
VALIDATION_FRACTION = 0.2
USE_FEATURE_CACHE = True
FORCE_REBUILD_FEATURES = False
USE_MODEL_CACHE = True
FORCE_RETRAIN_MODEL = False

cfg

## 0.1 Optional Dependency Check

Cell này chỉ kiểm tra package. Model optional vẫn có thể bỏ qua nếu chưa dùng.

In [ ]:
optional_packages = {
    "lightgbm": importlib.util.find_spec("lightgbm") is not None,
    "xgboost": importlib.util.find_spec("xgboost") is not None,
    "torch": importlib.util.find_spec("torch") is not None,
}
optional_packages

## 1. Build Train Features

Notebook sẽ cache feature CSV theo `RUN_MODE`, `time_bins`, và `limit`, ví dụ `outputs/features_train_smoke_tb20_limit5.csv`. Nếu file đã tồn tại, cell này đọc lại CSV thay vì build lại raw parquet. Đặt `FORCE_REBUILD_FEATURES = True` khi muốn tạo lại.

In [ ]:
repository = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
pipeline = ArielPreprocessFeaturePipeline(
    PreprocessConfig(target_time_bins=cfg["time_bins"]),
    FeatureConfig(),
)
builder = ArielDatasetBuilder(repository=repository, pipeline=pipeline)

limit_tag = "full" if cfg["limit"] is None else str(cfg["limit"])
feature_tag = f"{RUN_MODE}_tb{cfg['time_bins']}_limit{limit_tag}"
train_feature_path = OUTPUT_DIR / f"features_train_{feature_tag}.csv"

if USE_FEATURE_CACHE and train_feature_path.exists() and not FORCE_REBUILD_FEATURES:
    print(f"Loading cached train features: {train_feature_path}")
    train_features = pd.read_csv(train_feature_path)
else:
    print(f"Building train features from raw parquet: {train_feature_path}")
    train_result = builder.build_feature_csv(
        "train",
        train_feature_path,
        aggregate_observations=True,
        limit=cfg["limit"],
        on_error="raise",
    )
    train_features = train_result.features
print(train_features.shape)
train_features.head()

## 2. Join Targets

`align_features_and_targets` giữ lại chỉ các `planet_id` có cả feature và target. `groups` dùng để split/CV theo planet.

In [ ]:
targets = pd.read_csv(DATA_ROOT / "train.csv")
x_frame, y, groups, target_columns = align_features_and_targets(train_features, targets)

n_components = min(cfg["n_components"], y.shape[0], y.shape[1])
model_config = ModelConfig(
    n_components=n_components,
    random_state=RANDOM_STATE,
    calibrate_sigma=True,
)

print("X:", x_frame.shape)
print("Y:", y.shape)
print("n_components:", n_components)
x_frame.head()

## 3. Model/PCA Search

Search dùng `gaussian_nll` thấp nhất để chọn candidate. Danh sách mặc định tránh optional deps. Muốn thêm LightGBM/XGBoost, thêm vào `MODEL_CANDIDATES` sau khi cài optional deps.

In [ ]:
MODEL_CANDIDATES = (
    "bayesian_ridge",
    "ridge",
    "kernel_ridge",
    "extra_trees",
    "boosting",
    # "lightgbm",
    # "xgboost",
    # "br_lgbm_residual",
)
N_COMPONENTS_GRID = (20, 30, 40)

if cfg["run_search"]:
    search = hyperparameter_search(
        x_frame.to_numpy(dtype=float),
        y,
        model_names=MODEL_CANDIDATES,
        n_components_grid=N_COMPONENTS_GRID,
        base_config=ModelConfig(random_state=RANDOM_STATE),
        n_splits=CV_SPLITS,
        groups=groups,
        random_state=RANDOM_STATE,
    )
    cfg["model_name"] = search.best_candidate.model_name
    model_config = search.best_candidate.model_config
    search_table = pd.DataFrame([
        {
            "model_name": item.model_name,
            "n_components": item.model_config.n_components,
            **item.mean_metrics,
        }
        for item in search.candidates
    ]).sort_values("gaussian_nll")
    display(search_table)
else:
    print("Search disabled. Using:", cfg["model_name"], model_config)

model_tag = f"{cfg['model_name']}_pca{model_config.n_components}_{feature_tag}"
model_artifact_path = MODEL_DIR / f"{model_tag}.joblib"
metrics_path = MODEL_DIR / f"{model_tag}_metrics.json"
latest_artifact_path = MODEL_DIR / "model.joblib"

artifact = None
validation_metrics = None
if USE_MODEL_CACHE and model_artifact_path.exists() and not FORCE_RETRAIN_MODEL:
    print(f"Loading cached model artifact: {model_artifact_path}")
    artifact = joblib.load(model_artifact_path)
    validation_metrics = artifact.get("metrics", {})
else:
    print(f"Model will be trained and saved to: {model_artifact_path}")

## 4. Validation

Validation split theo `planet_id`. Metrics quan trọng nhất hiện tại: `gaussian_nll`, `rmse_mean`, và coverage 1σ/2σ.

In [ ]:
if artifact is not None:
    validation = None
    print("Using cached validation metrics from artifact.")
else:
    validation = train_model(
        x_frame.to_numpy(dtype=float),
        y,
        model_name=cfg["model_name"],
        model_config=model_config,
        validation_fraction=VALIDATION_FRACTION,
        groups=groups,
        random_state=RANDOM_STATE,
    )
    validation_metrics = validation.evaluation.as_dict()

print(json.dumps(validation_metrics, indent=2))

## 5. Optional Cross-Validation

Chạy CV sau khi smoke pass. Với model chậm như kernel/boosting/residual, CV có thể mất lâu vì mỗi PCA component là một model riêng.

In [ ]:
if artifact is not None:
    print("Cached model loaded. CV skipped. Set FORCE_RETRAIN_MODEL=True to recompute CV/training.")
elif cfg["run_cv"]:
    cv = cross_validate_model(
        x_frame.to_numpy(dtype=float),
        y,
        model_name=cfg["model_name"],
        model_config=model_config,
        n_splits=CV_SPLITS,
        groups=groups,
        random_state=RANDOM_STATE,
    )
    print(json.dumps(cv.mean_metrics, indent=2))
else:
    print("CV disabled.")

## 6. Refit Full Model And Save Artifact

Validation model chỉ dùng để đo metric. Nếu `REFIT_FULL=True`, artifact dùng model train lại trên toàn bộ rows để predict test.

In [ ]:
if artifact is None:
    final_model = validation.model
    if cfg["refit_full"]:
        final_model = refit_full_model(
            x_frame.to_numpy(dtype=float),
            y,
            model_name=cfg["model_name"],
            model_config=model_config,
        )

    artifact = {
        "model": final_model,
        "validation_model": validation.model,
        "feature_columns": list(x_frame.columns),
        "target_columns": target_columns,
        "metrics": validation_metrics,
        "run_config": cfg,
        "model_config": model_config,
        "feature_path": str(train_feature_path),
        "model_tag": model_tag,
    }

    joblib.dump(artifact, model_artifact_path)
    joblib.dump(artifact, latest_artifact_path)
    metrics_path.write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
    (MODEL_DIR / "metrics.json").write_text(json.dumps(validation_metrics, indent=2), encoding="utf-8")
else:
    print("Model artifact already loaded from cache.")

model_artifact_path

## 7. Build Test Features

Để tạo submission thật, đảm bảo `cfg["limit"] is None`. Nếu đang smoke test, submission chỉ chứa vài planet đầu.

In [ ]:
test_feature_path = OUTPUT_DIR / f"features_test_{feature_tag}.csv"

if USE_FEATURE_CACHE and test_feature_path.exists() and not FORCE_REBUILD_FEATURES:
    print(f"Loading cached test features: {test_feature_path}")
    test_features = pd.read_csv(test_feature_path)
else:
    print(f"Building test features from raw parquet: {test_feature_path}")
    test_result = builder.build_feature_csv(
        "test",
        test_feature_path,
        aggregate_observations=True,
        limit=cfg["limit"],
        on_error="raise",
    )
    test_features = test_result.features
print(test_features.shape)
test_features.head()

## 8. Generate Submission

Schema ưu tiên đọc từ `sample_submission.csv`. Nếu không có file đó, notebook dùng `target_columns` từ train.

In [ ]:
if artifact is None:
    artifact = joblib.load(model_artifact_path)
sample_path = DATA_ROOT / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path) if sample_path.exists() else None
schema = infer_submission_schema(
    sample_submission=sample_submission,
    target_columns=artifact["target_columns"],
)

submission = predict_submission(
    artifact["model"],
    test_features,
    feature_columns=artifact["feature_columns"],
    schema=schema,
)
save_submission(submission, OUTPUT_DIR / "submission.csv")
print(submission.shape)
submission.head()

## 9. Deep Learning Baselines

Các class DL đã có trong package: `CNN1DRegressor`, `TCNRegressor`, `LSTMRegressor`, `GRURegressor`, `TransformerSequenceRegressor`, `AutoencoderMLPRegressor`.

Chúng cần input dạng tensor `[samples, time, channels]`, không phải feature table `x_frame`. Vì vậy nên cache riêng light-curve tensors sau preprocess rồi train riêng. Không dùng trực tiếp trong submission workflow ở trên cho đến khi tensor cache được validate.